In [1]:
import sys
import os
from pathlib import Path
__file__ = os.getcwd()
ROOT_DIR = Path(__file__).parent.parent.parent
print(ROOT_DIR)

sys.path.insert(0, str(ROOT_DIR))

/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT


In [2]:
from videodeepsearch.core.app_state import (
    Appstate,
    get_app_state,
    get_context_file_sys,
    get_external_client,
    get_llm_instance,
    get_postgres_client,
    get_segment_milvus,
    get_storage_client,
    get_visual_image_client,
)

from videodeepsearch.tools.clients import *


app_state = Appstate()

In [3]:
# CONSTANT
MILVUS_HOST = "localhost"
MILVUS_PORT = 19530
MILVUS_URI = f"http://{MILVUS_HOST}:{MILVUS_PORT}"

IMAGE_COLLECTION_NAME = "image_milvus"
IMAGE_VISUAL_PARAM = {
    "type_config": "dense",
    "dimension": 512,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Image embeddings for video frames",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for image captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
IMAGE_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for image captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}
IMAGE_VISUAL_FIELD = "visual_embedding_field"
IMAGE_CAPTION_FIELD = "caption_embedding_field"
IMAGE_SPARSE_FIELD = "caption_sparse_embedding_field"


SEGMENT_CAPTION_COLLECTION_NAME = "segment_milvus"
SEGMENT_CAPTION_DENSE_PARAM = {
    "type_config": "dense",
    "dimension": 768,
    "metric_type": "COSINE",
    "index_type": "HNSW",
    "description": "Text embeddings for segment captions",
    "index_params": {"M": 64, "efConstruction": 100},
}
SEGMENT_CAPTION_SPARSE_PARAM = {
    "type_config": "sparse",
    "dimension": 1000000,  # ignored at runtime
    "metric_type": "BM25",
    "index_type": "SPARSE_INVERTED_INDEX",
    "description": "BM25 sparse index for segment captions",
    "index_params": {"inverted_index_algo": "DAAT_MAXSCORE"},
}

SEGMENT_DENSE_FIELD = "caption_embedding_field"
SEGMENT_SPARSE_FIELD = "caption_sparse_embedding_field"



In [4]:
image_milvus_client = ImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=IMAGE_COLLECTION_NAME,
    visual_param=IMAGE_VISUAL_PARAM,
    caption_param=IMAGE_CAPTION_DENSE_PARAM,
    sparse_param=IMAGE_CAPTION_SPARSE_PARAM,
    visual_ann_field=IMAGE_VISUAL_FIELD,
    caption_ann_field=IMAGE_CAPTION_FIELD,
    sparse_field=IMAGE_SPARSE_FIELD,
)

segment_caption_client = SegmentCaptionImageMilvusClient(
    uri=MILVUS_URI,
    collection_name=SEGMENT_CAPTION_COLLECTION_NAME,
    dense_param=SEGMENT_CAPTION_DENSE_PARAM,
    sparse_param=SEGMENT_CAPTION_SPARSE_PARAM,
    dense_field=SEGMENT_DENSE_FIELD,
    sparse_field=SEGMENT_SPARSE_FIELD,
)

await image_milvus_client.connect()
await segment_caption_client.connect()


app_state.image_milvus_client = image_milvus_client
app_state.segment_milvus_client = segment_caption_client

In [5]:
image_emebedding_settings = ImageEmbeddingSettings(
    model_name='open_clip',
    device='cuda',
    batch_size=32
)
text_emebedding_settings = TextEmbeddingSettings(
    model_name='mmbert',
    device='cuda',
    batch_size=32
)

img_txt_client = ImageEmbeddingClient(base_url='http://localhost:8003')
txt_client = TextEmbeddingClient(base_url='http://localhost:8005')


external_client = ExternalEncodeClient(img_text_client=img_txt_client, img_text_settings=image_emebedding_settings, txt_settings=text_emebedding_settings, txt_client=txt_client)

await external_client.connect()
app_state.external_client = external_client 

2025-12-03 20:12:27.933 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-image-embedding_model_loaded
2025-12-03 20:12:27.937 | INFO     | videodeepsearch.tools.clients.external.base:load_model:126 - service-text-embedding_model_loaded


In [6]:
MINIO_HOST = "localhost"       # use localhost when running locally
MINIO_PORT = 9000
MINIO_USER = "minioadmin"
MINIO_PASSWORD = "minioadmin"
MINIO_ACCESS_KEY = "minioadmin"
MINIO_SECRET_KEY = "minioadmin"
MINIO_SECURE = False  
MINIO_ENDPOINT = f"{MINIO_HOST}:{MINIO_PORT}"

storage_client = StorageClient(
    host=MINIO_HOST,
    port=MINIO_PORT,#type:ignore
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

storage_client._ensure_bucket('videotests')
app_state.minio_client = storage_client 

In [7]:

from videodeepsearch.core.config.llm_config import (
    llm_configs
)
from llama_index.llms.google_genai import GoogleGenAI
from dotenv import load_dotenv
import os
print(os.getcwd())
load_dotenv(dotenv_path='/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/videodeepsearch/.env')

for llm_config in llm_configs:
    agent_name = llm_config.agent_name
    app_state.llm_instance[agent_name] = GoogleGenAI(
        model=llm_config.model_name,
        generation_config=llm_config.generation_config
    )



/home/tinhanhnguyen/Desktop/HK7/Capstone/CAPSTONE_PROJECT/videodeepsearch/test/test_notebook2


In [9]:
from sqlalchemy import text

POSTGRE_USER = "prefect"
POSTGRE_PASSWORD = "prefect"
POSTGRE_HOST = "localhost"       # use localhost for local setup
POSTGRE_PORT = 5432
POSTGRE_DB = "prefect"

POSTGRE_DATABASE_URL = (
    f"postgresql+asyncpg://{POSTGRE_USER}:{POSTGRE_PASSWORD}"
    f"@{POSTGRE_HOST}:{POSTGRE_PORT}/{POSTGRE_DB}"
)
postgres_client = PostgresClient(
    database_url=POSTGRE_DATABASE_URL
)
async with postgres_client.get_session() as session:
    result = await session.execute(text("SELECT version();"))
    version = result.scalar_one()
    print(f"🗄️ PostgreSQL connection successful.\n   Version: {version}")

app_state.postgres_client = postgres_client

🗄️ PostgreSQL connection successful.
   Version: PostgreSQL 14.20 (Debian 14.20-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


In [10]:
from videodeepsearch.agent.worker.definition import  WORKER_AGENT, ORCHESTRATOR_AGENT, PLANNER_AGENT, GREETER_AGENT
from videodeepsearch.agent.base import get_global_agent_registry
from videodeepsearch.tools.implementation.llm.tool import enhance_textual_query, enhance_visual_query
from videodeepsearch.tools.implementation.persist.orc_tool import update_video_context_orc_agent, orc_synthesize_final_findings
from videodeepsearch.tools.implementation.persist.worker_tool import worker_persist_evidence
from videodeepsearch.tools.implementation.scan.tool import get_segments, get_image, extract_frames_by_time_window, extract_frame_time
from videodeepsearch.tools.implementation.search.tool import get_images_from_caption_query, get_images_from_multimodal_query, get_images_from_visual_query, get_segments_from_event_query
from videodeepsearch.tools.implementation.util.tool import get_related_asr_from_image, get_related_asr_from_video_id
from videodeepsearch.tools.implementation.view.worker_tool import worker_view_all_data_handle, worker_view_my_evidence, worker_view_results, worker_view_statistics

from videodeepsearch.agent.worker.agent_as_tool import run_planning_agent_as_tool


In [14]:

from videodeepsearch.agent.context.management import FileSystemContextStore
from videodeepsearch.agent.worker.agent_as_tool import (
    run_planning_agent_as_tool,
    running_worker_agent_as_tools,
    running_orchestrator_agent_as_tools,
)

In [16]:
store_folder = './context_persistance'
import os
os.makedirs(store_folder, exist_ok=True)
context_management = FileSystemContextStore(storage_dir=store_folder)

In [ ]:
import asyncio
from typing import AsyncGenerator
from uuid import uuid4
from videodeepsearch.agent.base import get_global_agent_registry
from llama_index.core.tools import FunctionTool
from llama_index.core.llms import ChatMessage, MessageRole
from llama_index.core.agent.workflow import AgentOutput, AgentStream


class SimpleChatSession:
    """Simple chat session manager for Jupyter notebooks"""
    
    def __init__(
        self,
        session_id: str | None = None,
        user_id: str = "default_user",
        list_video_id: list[str] | None = None
    ):
        self.session_id = session_id or str(uuid4())
        self.user_id = user_id
        self.list_video_id = list_video_id or []
        self.chat_history: list[ChatMessage] = []
        
        print(f"📱 Session started: {self.session_id}")
        print(f"👤 User ID: {self.user_id}")
        print(f"🎥 Video IDs: {self.list_video_id}")
        print("-" * 60)
    
    async def send_message(self, message: str, stream: bool = True):
        print(f"\nYou: {message}")
        print("-" * 60)

        agent_registry = get_global_agent_registry()
        orchestration_partial_params = {
            'session_id': self.session_id,
            'user_id': self.user_id,
            'list_video_id': self.list_video_id,
            'user_original_user_message': message
        }

        orchestrator_as_tools = FunctionTool.from_defaults(
            async_fn=running_orchestrator_agent_as_tools,
            partial_params=orchestration_partial_params
        )


        